# Week 1 — Distributed Data Processing & Feature Engineering (FIXED)

Project 3: Context-Aware Neural Recommendation Engine (H&M dataset).

**What changed from the first version** (search for `FIX` in this notebook):

| # | Fix | Why |
|---|-----|-----|
| 1 | Train/test **interaction rows are now saved** to parquet | Week 2 needs them to train the two-tower model |
| 2 | User/item **vocabularies built from ALL customers / ALL articles**, index `0` = unknown | Handles cold-start users and items |
| 3 | `article_id` read as **string** | Keeps leading zeros (`0108775015`), matches real H&M IDs |
| 4 | Transactions read with an **explicit schema** and converted to parquet once | Much faster; no re-scanning the 3.5 GB CSV |
| 5 | Category vocabularies + top-12 fallback **saved to files**; `kaggle.json` removed from Drive | Week 2/3 can reuse them; API key stays private |

Run it top to bottom with **Runtime → Run all**. If the CSVs are already in your Drive folder, nothing is downloaded again.

## 1. Setup

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
!mkdir -p /content/drive/MyDrive/hm_project
%cd /content/drive/MyDrive/hm_project

/content/drive/MyDrive/hm_project


### Download data (only if missing)
If the CSVs are already in the Drive folder this cell does nothing. Otherwise it asks you to upload `kaggle.json`.

In [24]:
import os

needed = ["articles.csv", "customers.csv", "transactions_train.csv"]
missing = [f for f in needed if not os.path.exists(f)]
print("Missing files:", missing)

if missing:
    from google.colab import files
    files.upload()  # choose kaggle.json
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    for f in missing:
        !kaggle competitions download -c h-and-m-personalized-fashion-recommendations -f {f}
    !unzip -o -q "*.zip"
    !rm -f *.zip

# FIX 5: never keep the Kaggle API key inside the project folder
!rm -f kaggle.json
!ls -lh

Missing files: []
total 3.5G
-rw------- 1 root root  35M Nov  5 10:16  articles.csv
drwx------ 2 root root 4.0K Nov 17 11:34  customer_feats.parquet
-rw------- 1 root root 198M Nov  5 10:16  customers.csv
-rw------- 1 root root  168 Nov 17 11:36  fallback_top12.json
drwx------ 2 root root 4.0K Nov 17 11:27  interactions_test.parquet
drwx------ 2 root root 4.0K Nov 17 11:26  interactions_train.parquet
drwx------ 2 root root 4.0K Nov 17 11:35  item_popularity.parquet
drwx------ 2 root root 4.0K Nov 17 11:31  item_table.parquet
drwx------ 2 root root 4.0K Nov 17 11:31  item_vocab.parquet
-rw------- 1 root root   70 Nov  4 17:21 'kaggle (1).json'
-rw------- 1 root root   70 Nov  5 05:52 'kaggle (2).json'
-rw------- 1 root root   70 Nov  5 07:00 'kaggle (3).json'
-rw------- 1 root root   70 Nov  5 10:00 'kaggle (4).json'
-rw------- 1 root root   70 Nov  5 10:16 'kaggle (5).json'
drwx------ 2 root root 4.0K Nov 17 11:17  transactions.parquet
-rw------- 1 root root 3.3G Nov  5 10:16  transact

In [25]:
!pip install -q pyspark

In [26]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType

spark = (
    SparkSession.builder
    .appName("hm_project")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)

## 2. Load data

**FIX 3 + FIX 4:** explicit schema, `article_id` as **string** (keeps the leading zero).
The 3.5 GB CSV is converted to parquet **once**; afterwards we always read the fast parquet copy.

In [27]:
txn_schema = StructType([
    StructField("t_dat", DateType(), True),
    StructField("customer_id", StringType(), True),
    StructField("article_id", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("sales_channel_id", IntegerType(), True),
])

if not os.path.exists("transactions.parquet"):
    (spark.read.csv("transactions_train.csv", header=True, schema=txn_schema)
         .write.mode("overwrite").parquet("transactions.parquet"))

transactions = spark.read.parquet("transactions.parquet")
transactions.printSchema()
transactions.show(5, truncate=False)   # article_id should start with 0

root
 |-- t_dat: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)

+----------+----------------------------------------------------------------+----------+--------------------+----------------+
|t_dat     |customer_id                                                     |article_id|price               |sales_channel_id|
+----------+----------------------------------------------------------------+----------+--------------------+----------------+
|2019-11-29|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|0706016003|0.025288135593220337|2               |
|2019-11-29|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|0706016001|0.025288135593220337|2               |
|2019-11-29|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|0682236013|0.018983050847457626|2               |
|2019-11-29|aaa78c87aacba9

In [28]:
customers = spark.read.csv("customers.csv", header=True, inferSchema=True)

# no inferSchema -> every column is a string, so article_id keeps its leading zero
articles = spark.read.csv("articles.csv", header=True, multiLine=True, escape='"')

print("Customers in customers.csv:", customers.count())   # expect 1,371,980
print("Articles in articles.csv:", articles.count())      # expect 105,542
articles.select("article_id", "prod_name").show(3, truncate=False)

Customers in customers.csv: 1371980
Articles in articles.csv: 105542
+----------+-------------+
|article_id|prod_name    |
+----------+-------------+
|0108775015|Strap top    |
|0108775044|Strap top    |
|0108775051|Strap top (1)|
+----------+-------------+
only showing top 3 rows


## 3. Explore & check data quality

In [29]:
print("Transactions:", transactions.count())
transactions.select(F.min("t_dat"), F.max("t_dat")).show()

transactions.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in transactions.columns]
).show()

print("Customers who bought:", transactions.select("customer_id").distinct().count())
print("Articles bought:", transactions.select("article_id").distinct().count())

Transactions: 31788324
+----------+----------+
|min(t_dat)|max(t_dat)|
+----------+----------+
|2018-09-20|2020-09-22|
+----------+----------+

+-----+-----------+----------+-----+----------------+
|t_dat|customer_id|article_id|price|sales_channel_id|
+-----+-----------+----------+-----+----------------+
|    0|          0|         0|    0|               0|
+-----+-----------+----------+-----+----------------+

Customers who bought: 1362281
Articles bought: 104547


In [30]:
customers.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in customers.columns]
).show()
customers.select("age").describe().show()

+-----------+------+------+------------------+----------------------+-----+-----------+
|customer_id|    FN|Active|club_member_status|fashion_news_frequency|  age|postal_code|
+-----------+------+------+------------------+----------------------+-----+-----------+
|          0|895050|907576|              6062|                 16009|15861|          0|
+-----------+------+------+------------------+----------------------+-----+-----------+

+-------+-----------------+
|summary|              age|
+-------+-----------------+
|  count|          1356119|
|   mean|  36.386964565794|
| stddev|14.31362798162793|
|    min|               16|
|    max|               99|
+-------+-----------------+



### Cold-start check

In [31]:
no_purchase = customers.join(transactions.select("customer_id").distinct(), on="customer_id", how="left_anti")
print("Customers with no purchases:", no_purchase.count())

unsold = articles.join(transactions.select("article_id").distinct(), on="article_id", how="left_anti")
print("Articles never bought:", unsold.count())

Customers with no purchases: 9699
Articles never bought: 995


## 4. Handle missing values (customers)

In [32]:
median_age = int(customers.approxQuantile("age", [0.5], 0.01)[0])
print("Median age:", median_age)

customers_clean = (
    customers
    .withColumn("age_missing", F.col("age").isNull().cast("int"))
    .withColumn("fashion_news_frequency", F.upper(F.col("fashion_news_frequency")))
    .fillna({
        "age": median_age,
        "FN": 0,
        "Active": 0,
        "club_member_status": "UNKNOWN",
        "fashion_news_frequency": "NONE",
    })
)

customers_clean.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in customers_clean.columns]
).show()
customers_clean.groupBy("fashion_news_frequency").count().show()

Median age: 32
+-----------+---+------+------------------+----------------------+---+-----------+-----------+
|customer_id| FN|Active|club_member_status|fashion_news_frequency|age|postal_code|age_missing|
+-----------+---+------+------------------+----------------------+---+-----------+-----------+
|          0|  0|     0|                 0|                     0|  0|          0|          0|
+-----------+---+------+------------------+----------------------+---+-----------+-----------+

+----------------------+------+
|fashion_news_frequency| count|
+----------------------+------+
|               MONTHLY|   842|
|                  NONE|893722|
|             REGULARLY|477416|
+----------------------+------+



## 5. Train / test split (by time)
Last week of data (from 2020-09-16) is the test set — same as the Kaggle evaluation.

In [33]:
cutoff = "2020-09-16"
train = transactions.filter(F.col("t_dat") < cutoff)
test = transactions.filter(F.col("t_dat") >= cutoff)
print("Train rows:", train.count())
print("Test rows:", test.count())

Train rows: 31548013
Test rows: 240311


## 6. Contextual features (time of purchase, recency, popularity over time)

In [34]:
train = (
    train
    .withColumn("month", F.month("t_dat"))
    .withColumn("day_of_week", F.dayofweek("t_dat"))
    .withColumn("days_ago", F.datediff(F.lit(cutoff).cast("date"), F.col("t_dat")))
)

In [35]:
customer_feats = (
    train.groupBy("customer_id")
    .agg(
        F.count("*").alias("n_purchases"),
        F.countDistinct("article_id").alias("n_unique_items"),
        F.avg("price").alias("avg_price"),
        F.min("days_ago").alias("days_since_last_purchase"),
    )
)
customer_feats.show(5)

+--------------------+-----------+--------------+--------------------+------------------------+
|         customer_id|n_purchases|n_unique_items|           avg_price|days_since_last_purchase|
+--------------------+-----------+--------------+--------------------+------------------------+
|3f2117aaebaf090d8...|         12|            12| 0.02419209039548022|                     202|
|9c9de8bc2af06bd45...|         21|            20|0.024971751412429376|                      36|
|159ca60dd684a6814...|         23|            22| 0.02530876934414148|                      21|
|25106d25ffcd8a629...|         61|            54| 0.03677132536815782|                      47|
|7a89e550ccd039913...|         98|            80| 0.02576150121065375|                      76|
+--------------------+-----------+--------------+--------------------+------------------------+
only showing top 5 rows


In [36]:
# popularity over time: last 7 days, last 28 days, all time
item_popularity = (
    train.groupBy("article_id")
    .agg(
        F.sum(F.when(F.col("days_ago") <= 7, 1).otherwise(0)).alias("purchases_last_7d"),
        F.sum(F.when(F.col("days_ago") <= 28, 1).otherwise(0)).alias("purchases_last_28d"),
        F.count("*").alias("purchases_all_time"),
    )
)
top_items = item_popularity.orderBy(F.desc("purchases_last_28d")).limit(12)
top_items.show()

+----------+-----------------+------------------+------------------+
|article_id|purchases_last_7d|purchases_last_28d|purchases_all_time|
+----------+-----------------+------------------+------------------+
|0751471001|              607|              2758|             17737|
|0706016001|              453|              2408|             49958|
|0915526001|              443|              2383|              2621|
|0751471043|              435|              2328|              2570|
|0918292001|              546|              2327|              3241|
|0915529003|              450|              2177|              2202|
|0896152002|              381|              2028|              2526|
|0898694001|              380|              1836|              1836|
|0916468003|               90|              1822|              2669|
|0448509014|              609|              1807|             18726|
|0863595006|              376|              1767|              2418|
|0915526002|              219|    

## 7. Vocabularies for embeddings

**FIX 2:** built from **all** customers and **all** articles (not just people/items in train),
so cold-start users and never-bought items still get an index and features.
Index **0 is reserved for unknown** (row_number starts at 1).

In [37]:
user_vocab = (
    customers_clean.select("customer_id").distinct()
    .withColumn("user_idx", F.row_number().over(Window.orderBy("customer_id")))
)
item_vocab = (
    articles.select("article_id").distinct()
    .withColumn("item_idx", F.row_number().over(Window.orderBy("article_id")))
)
print("Users:", user_vocab.count())
print("Items:", item_vocab.count())

Users: 1371980
Items: 105542


In [38]:
item_cat_cols = ["product_type_name", "colour_group_name", "department_name", "garment_group_name"]

user_table = (
    user_vocab
    .join(customers_clean.select("customer_id", "age", "club_member_status",
                                 "fashion_news_frequency", "age_missing"),
          on="customer_id", how="left")
    .join(customer_feats, on="customer_id", how="left")
    .withColumn("is_new_user", F.col("n_purchases").isNull().cast("int"))
    .fillna({"n_purchases": 0, "n_unique_items": 0, "avg_price": 0.0,
             "days_since_last_purchase": 999})   # 999 = never bought
)

item_table = (
    item_vocab
    .join(articles.select("article_id", *item_cat_cols), on="article_id", how="left")
    .join(item_popularity, on="article_id", how="left")
    .withColumn("is_new_item", F.col("purchases_all_time").isNull().cast("int"))
    .fillna({"purchases_last_7d": 0, "purchases_last_28d": 0, "purchases_all_time": 0})
    .fillna("UNKNOWN", subset=item_cat_cols)
)

user_table.groupBy("is_new_user").count().show()
item_table.groupBy("is_new_item").count().show()

+-----------+-------+
|is_new_user|  count|
+-----------+-------+
|          1|  15271|
|          0|1356709|
+-----------+-------+

+-----------+------+
|is_new_item| count|
+-----------+------+
|          1|  1662|
|          0|103880|
+-----------+------+



## 8. Training data for Week 2

**FIX 1:** save the actual purchase rows (user, item, context). This is what the two-tower model trains on.

In [39]:
interactions_train = (
    train
    .join(user_vocab, on="customer_id", how="left")
    .join(item_vocab, on="article_id", how="left")
    .fillna({"user_idx": 0, "item_idx": 0})
    .select("user_idx", "item_idx", "t_dat", "month", "day_of_week",
            "days_ago", "price", "sales_channel_id")
)

interactions_test = (
    test
    .join(user_vocab, on="customer_id", how="left")
    .join(item_vocab, on="article_id", how="left")
    .fillna({"user_idx": 0, "item_idx": 0})
    .select("user_idx", "item_idx", "t_dat", "price", "sales_channel_id")
)

interactions_train.show(5)

+--------+--------+----------+-----+-----------+--------+--------------------+----------------+
|user_idx|item_idx|     t_dat|month|day_of_week|days_ago|               price|sales_channel_id|
+--------+--------+----------+-----+-----------+--------+--------------------+----------------+
|  342403|   15148|2018-12-13|   12|          5|     643|0.008457627118644067|               1|
|  342403|   15148|2018-12-13|   12|          5|     643|0.008457627118644067|               1|
|  614942|   63473|2019-07-28|    7|          1|     416|0.025406779661016947|               1|
|  615159|   55779|2019-07-28|    7|          1|     416|0.013542372881355931|               1|
|  615175|   54466|2019-07-28|    7|          1|     416|0.010152542372881357|               2|
+--------+--------+----------+-----+-----------+--------+--------------------+----------------+
only showing top 5 rows


## 9. Save everything

In [40]:
interactions_train.write.mode("overwrite").parquet("interactions_train.parquet")
interactions_test.write.mode("overwrite").parquet("interactions_test.parquet")
user_table.write.mode("overwrite").parquet("user_table.parquet")
item_table.write.mode("overwrite").parquet("item_table.parquet")
user_vocab.write.mode("overwrite").parquet("user_vocab.parquet")
item_vocab.write.mode("overwrite").parquet("item_vocab.parquet")
customer_feats.write.mode("overwrite").parquet("customer_feats.parquet")
item_popularity.write.mode("overwrite").parquet("item_popularity.parquet")
print("Saved parquet files")

Saved parquet files


In [41]:
# FIX 5: save category vocabularies + popular-items fallback as small JSON files
import json

vocabularies = {}
for c in item_cat_cols:
    vocabularies[c] = sorted(r[0] for r in item_table.select(c).distinct().collect())
for c in ["club_member_status", "fashion_news_frequency"]:
    vocabularies[c] = sorted(r[0] for r in user_table.select(c).distinct().collect())

with open("vocabularies.json", "w") as f:
    json.dump(vocabularies, f, indent=2)

top12 = [r["article_id"] for r in top_items.collect()]
with open("fallback_top12.json", "w") as f:
    json.dump(top12, f)

for k, v in vocabularies.items():
    print(k, len(v))
print("Top 12 fallback:", top12)

product_type_name 131
colour_group_name 50
department_name 250
garment_group_name 21
club_member_status 4
fashion_news_frequency 3
Top 12 fallback: ['0751471001', '0706016001', '0915526001', '0751471043', '0918292001', '0915529003', '0896152002', '0898694001', '0916468003', '0448509014', '0863595006', '0915526002']


## 10. Quick check of the outputs

In [42]:
for name in ["interactions_train", "interactions_test", "user_table", "item_table"]:
    df = spark.read.parquet(f"{name}.parquet")
    print(f"{name}: {df.count():,} rows")
    df.printSchema()

!du -sh *.parquet *.json

interactions_train: 31,548,013 rows
root
 |-- user_idx: integer (nullable = true)
 |-- item_idx: integer (nullable = true)
 |-- t_dat: date (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- days_ago: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)

interactions_test: 240,311 rows
root
 |-- user_idx: integer (nullable = true)
 |-- item_idx: integer (nullable = true)
 |-- t_dat: date (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)

user_table: 1,371,980 rows
root
 |-- customer_id: string (nullable = true)
 |-- user_idx: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- club_member_status: string (nullable = true)
 |-- fashion_news_frequency: string (nullable = true)
 |-- age_missing: integer (nullable = true)
 |-- n_purchases: long (nullable = true)
 |-- n_unique_items: long (nullable = true)
 |-- a

## Output files (hand-off to Week 2)

All files are in `MyDrive/hm_project/`. **Index 0 = unknown user/item.**

| File | Columns | Use |
|------|---------|-----|
| `interactions_train.parquet` | user_idx, item_idx, t_dat, month, day_of_week, days_ago, price, sales_channel_id | Model training rows (before 2020-09-16) |
| `interactions_test.parquet` | user_idx, item_idx, t_dat, price, sales_channel_id | Evaluation (Recall@K, NDCG), last week |
| `user_table.parquet` | customer_id, user_idx, age, club_member_status, fashion_news_frequency, age_missing, n_purchases, n_unique_items, avg_price, days_since_last_purchase, is_new_user | Query tower features |
| `item_table.parquet` | article_id, item_idx, product_type_name, colour_group_name, department_name, garment_group_name, purchases_last_7d, purchases_last_28d, purchases_all_time, is_new_item | Candidate tower features |
| `user_vocab.parquet` / `item_vocab.parquet` | id ↔ idx | Map real IDs to embedding indexes |
| `vocabularies.json` | category value lists | StringLookup vocabularies |
| `fallback_top12.json` | 12 article_ids | Cold-start fallback recommendations |